# Synthetic Data Generation

## 1. Config

In [ ]:
print("===========================| Config started... |===\n")

CONFIGS = {
    "qwen7b_zeroshot": {"model": "Qwen/Qwen2.5-7B-Instruct", "strategy": "zero_shot_icl"},
    "qwen7b_fewshot_1r": {"model": "Qwen/Qwen2.5-7B-Instruct", "strategy": "few_shot_icl_1r"},
    "qwen7b_fewshot_5r": {"model": "Qwen/Qwen2.5-7B-Instruct", "strategy": "few_shot_icl_5r"},
    "qwen7b_fewshot_10r": {"model": "Qwen/Qwen2.5-7B-Instruct", "strategy": "few_shot_icl_10r"},
    "qwen7b_lora_10": {"model": "Qwen/Qwen2.5-7B-Instruct", "strategy": "lora", "dataset": "dataset/cct_ft_10pct.json"},
    "qwen7b_lora_50": {"model": "Qwen/Qwen2.5-7B-Instruct", "strategy": "lora", "dataset": "dataset/cct_ft_50pct.json"},
    "mistral7b_zeroshot": {"model": "mistralai/Mistral-7B-Instruct-v0.3", "strategy": "zero_shot_icl"},
    "mistral7b_fewshot_1r": {"model": "mistralai/Mistral-7B-Instruct-v0.3", "strategy": "few_shot_icl_1r"},
    "mistral7b_fewshot_5r": {"model": "mistralai/Mistral-7B-Instruct-v0.3", "strategy": "few_shot_icl_5r"},
    "mistral7b_fewshot_10r": {"model": "mistralai/Mistral-7B-Instruct-v0.3", "strategy": "few_shot_icl_10r"},
    "mistral7b_lora_10": {"model": "mistralai/Mistral-7B-Instruct-v0.3", "strategy": "lora", "dataset": "dataset/cct_ft_10pct.json"},
    "mistral7b_lora_50": {"model": "mistralai/Mistral-7B-Instruct-v0.3", "strategy": "lora", "dataset": "dataset/cct_ft_50pct.json"},
}

PARAMS = {
    "config": "qwen7b_zeroshot",
    "seed": 0,
    "num_samples": 10000,
    "batch_size": 8,
    "max_new_tokens": 1024,
    "temperature": 0.8,
    "top_p": 0.95,
    "repetition_penalty": 1.05,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "learning_rate": 1e-4,
    "epochs": 1,
}

print("\n===========================| Config completed. |===")

## 2. Setup

In [ ]:
print("===========================| Setup started... |===\n")

import os
import subprocess

def select_gpu():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,nounits,noheader"],
        capture_output=True, text=True
    )
    free_memories = [int(x) for x in result.stdout.strip().split("\n")]
    best = free_memories.index(max(free_memories))
    print(f"Selected GPU {best} ({max(free_memories)} MiB free)")
    os.environ["CUDA_VISIBLE_DEVICES"] = str(best)

select_gpu()

import json
import torch
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    set_seed,
)
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
from datasets import Dataset, load_dataset

def get_config():
    config_name = PARAMS["config"]
    seed = PARAMS["seed"]
    if config_name not in CONFIGS:
        raise ValueError(f"Config '{config_name}' not found. Available: {list(CONFIGS.keys())}")
    config = {**PARAMS, **CONFIGS[config_name]}
    config["name"] = config_name
    config["run_name"] = f"{config_name}_{seed}"
    config["start_idx"] = seed * config["num_samples"]
    
    print(f"Experiment: {config['run_name']}")
    print(f"- Strategy: {config['strategy']}")
    print(f"- Model: {config['model']}")
    print(f"- Samples: {config['start_idx']} → {config['start_idx'] + config['num_samples'] - 1}")
    return config

config = get_config()

if not torch.cuda.is_available():
    raise SystemExit("No GPU available. Exiting...")

set_seed(config["seed"])
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(config["seed"])

Path("outputs").mkdir(exist_ok=True)

print("\n===========================| Setup completed. |===")

## 3. Load Model

In [ ]:
print("===========================| Load Model started... |===\n")

def load_model_and_tokenizer(model_name):
    print(f"Loading {model_name}...")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto",
        trust_remote_code=True,
    )
    
    return model, tokenizer

model, tokenizer = load_model_and_tokenizer(config["model"])

print("\n===========================| Load Model completed. |===")

## 4. Load Data

In [ ]:
print("===========================| Load Data started... |===\n")

OUTPUT_DIR = Path("dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "pointe77/credit-card-transaction"
SEED = 42

datasets_config_meta = [
    ("cct_ft_100pct.json",  "Fine-Tuning 100%",        None, None),
    ("cct_ft_50pct.json",   "Fine-Tuning 50%",         0.50, None),
    ("cct_ft_10pct.json",   "Fine-Tuning 10%",         0.10, None),
    ("cct_icl_fs_10r.json", "Few-shot ICL 10 records", None, 10),
    ("cct_icl_fs_5r.json",  "Few-shot ICL 5 records",  None, 5),
    ("cct_icl_fs_1r.json",  "Few-shot ICL 1 record",   None, 1),
]

def ensure_datasets(output_dir, dataset_name, datasets_meta, seed):
    if all((output_dir / filename).exists() for filename, *_ in datasets_meta):
        print("All dataset files already exist, skipping download.\n")
        return

    print("Downloading dataset...")
    train_df = load_dataset(dataset_name, split="train").to_pandas()
    train_df = train_df.drop(columns=["trans_date_trans_time"])
    train_df["zip"]           = train_df["zip"].fillna("").astype(str)
    train_df["merch_zipcode"] = train_df["merch_zipcode"].apply(lambda x: "" if pd.isna(x) else str(int(x)))
    print(f"Total records: {len(train_df):,}\n")

    for filename, description, frac, n in datasets_meta:
        filepath = output_dir / filename
        if filepath.exists():
            print(f"  {description:30} -> {filename} (skipped)")
            continue
        if frac is not None:
            df = train_df.sample(frac=frac, random_state=seed)
        elif n is not None:
            df = train_df.sample(n=n, random_state=seed)
        else:
            df = train_df
        df.to_json(filepath, orient="records", indent=2)
        print(f"  {description:30} {len(df):5,} rows -> {filename}")

def load_training_data(config):
    if config["strategy"] not in ["lora"]:
        print("ICL mode: no training data needed")
        return None

    dataset_path = OUTPUT_DIR / Path(config["dataset"]).name
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset not found: {dataset_path}")

    train_data = json.loads(dataset_path.read_text(encoding="utf-8"))
    print(f"Training data loaded: {len(train_data)} samples from {dataset_path.name}")
    return train_data

ensure_datasets(OUTPUT_DIR, DATASET_NAME, datasets_config_meta, SEED)
train_data = load_training_data(config)

print("\n===========================| Load Data completed. |===")

## 5. Build Prompt

In [ ]:
print("===========================| Build Prompt started... |===\n")

EXAMPLES_DISCLAIMER = "The following are real records shown only to illustrate the expected format and value ranges.\nYou MUST NOT copy any field values from these examples. Generate completely different data.\n"

def load_prompt(config):
    raw = Path("prompt.md").read_text(encoding="utf-8").strip()
    parts = raw.split("---USER---")
    system_prompt = parts[0].strip()
    user_prompt = parts[1].strip() + "\n"
    if config["strategy"].startswith("few_shot_icl"):
        suffix = config["strategy"].replace("few_shot_icl_", "")
        raw_ex = Path(f"dataset/cct_icl_fs_{suffix}.json").read_text(encoding="utf-8").strip()
        user_prompt += "\n# EXAMPLES\n" + EXAMPLES_DISCLAIMER + "\n".join(raw_ex.splitlines()[1:-1]).strip() + "\n"
    print(f"System prompt ({len(system_prompt)} chars)")
    print(f"User prompt ({len(user_prompt)} chars)")
    return system_prompt, user_prompt

def build_prompt_tokens(model, tokenizer, system_prompt, user_prompt, config):
    messages = [
        {"role": "system",    "content": system_prompt},
        {"role": "user",      "content": user_prompt},
        {"role": "assistant", "content": "```json\n"},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=False,
        continue_final_message=True,
    )
    print("-" * 80)
    print("PROMPT:\n", text)
    print("-" * 80)
    return tokenizer(text, return_tensors="pt").to(next(model.parameters()).device)

system_prompt, user_prompt = load_prompt(config)
prompt_tokens = build_prompt_tokens(model, tokenizer, system_prompt, user_prompt, config)

print("\n===========================| Build Prompt completed. |===")

## 6. Setup Fine-Tuning

In [ ]:
print("===========================| Setup Fine-Tuning started... |===\n")

def setup_peft_model(model, config):
    output_path = f"outputs/{config['name']}_model"

    if config["strategy"] in ["lora"] and Path(output_path).exists():
        print(f"Found existing adapter at '{output_path}', skipping training.")
        config["_skip_training"] = True
        return model

    config["_skip_training"] = False

    if config["strategy"] == "lora":
        peft_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=config["lora_r"],
            lora_alpha=config["lora_alpha"],
            lora_dropout=config["lora_dropout"],
            target_modules="all-linear",
        )
    else:
        print("ICL mode: no fine-tuning needed")
        return model

    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    return model


def tokenize_single(example, system_prompt, user_prompt, tokenizer):
    example = dict(example)
    completion = "```json\n" + json.dumps(example, ensure_ascii=False) + "\n```"
    messages_full = [
        {"role": "system",    "content": system_prompt},
        {"role": "user",      "content": user_prompt},
        {"role": "assistant", "content": completion},
    ]
    messages_prompt = messages_full[:-1]

    full_text   = tokenizer.apply_chat_template(messages_full,   tokenize=False, add_generation_prompt=False)
    prompt_text = tokenizer.apply_chat_template(messages_prompt, tokenize=False, add_generation_prompt=True)
    prompt_text += "```json\n"

    full_ids   = tokenizer(full_text,   add_special_tokens=False)["input_ids"]
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

    prompt_len = len(prompt_ids)
    labels = [-100] * prompt_len + full_ids[prompt_len:]

    return {"input_ids": full_ids, "labels": labels}


def prepare_trainer(model, tokenizer, train_data, config, system_prompt, user_prompt):
    if train_data is None:
        return None

    dataset = Dataset.from_list(train_data)
    tokenized_dataset = dataset.map(
        lambda ex: tokenize_single(ex, system_prompt, user_prompt, tokenizer),
        remove_columns=dataset.column_names,
    )

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=f"outputs/{config['name']}_training",
            num_train_epochs=config["epochs"],
            per_device_train_batch_size=config["batch_size"],
            learning_rate=config["learning_rate"],
            gradient_checkpointing=True,
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            save_strategy="no",
            report_to="none",
        ),
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, padding=True),
    )
    return trainer


model  = setup_peft_model(model, config)
trainer = prepare_trainer(model, tokenizer, train_data, config, system_prompt, user_prompt)

print("\n===========================| Setup Fine-Tuning completed. |===")

## 7. Train

In [ ]:
print("===========================| Train started... |===\n")

def train_model(trainer, model, tokenizer, config):
    output_path = f"outputs/{config['name']}_model"

    if config.get("_skip_training"):
        print(f"Loading pre-trained adapter from '{output_path}'...")
        model = PeftModel.from_pretrained(model, output_path)
        model.eval()
        return model

    if trainer is None:
        print("ICL mode: no training needed")
        return model

    print("Training...")
    trainer.train()

    model.save_pretrained(output_path)
    tokenizer.save_pretrained(output_path)
    print(f"Model saved to: {output_path}")

    return model

model = train_model(trainer, model, tokenizer, config)

print("\n===========================| Train completed. |===")

## 8. Generate Synthetic Data

In [ ]:
print("===========================| Generate Synthetic Data started... |===\n")

REQUIRED_FIELDS = {
    "Unnamed: 0": int,
    "cc_num": int,
    "merchant": str,
    "category": str,
    "amt": float,
    "first": str,
    "last": str,
    "gender": str,
    "street": str,
    "city": str,
    "state": str,
    "zip": str,
    "lat": float,
    "long": float,
    "city_pop": int,
    "job": str,
    "dob": str,
    "trans_num": str,
    "unix_time": int,
    "merch_lat": float,
    "merch_long": float,
    "is_fraud": int,
    "merch_zipcode": str,
}

NULLABLE_FIELDS = {
    "merch_zipcode"
}

def is_valid_record(record):
    if not isinstance(record, dict):
        return False
    if set(record.keys()) != set(REQUIRED_FIELDS.keys()):
        return False
    for field, expected_type in REQUIRED_FIELDS.items():
        value = record.get(field)
        if field in NULLABLE_FIELDS:
            if value is not None and not isinstance(value, expected_type):
                return False
            continue
        if value is None or (isinstance(value, str) and not value.strip()):
            return False
        if not isinstance(value, expected_type):
            if expected_type is float and isinstance(value, int):
                continue
            return False
    return True

def parse_json_response(response):
    try:
        return json.loads(response.strip().split("```")[0].strip())
    except json.JSONDecodeError:
        return None

def generate_samples(model, tokenizer, prompt_tokens, config, output_file):
    model.eval()
    predictions = []
    failed_responses = []
    prompt_token_length = prompt_tokens["input_ids"].shape[1]
    debug_file = output_file.replace(".json", "_debug.json")
    target = config["num_samples"]
    print(f"Generating {target} valid samples...")
    with torch.inference_mode():
        with tqdm(total=target, desc="Generating") as pbar:
            while len(predictions) < target:
                outputs = model.generate(
                    **prompt_tokens,
                    max_new_tokens=config["max_new_tokens"],
                    temperature=config["temperature"],
                    top_p=config["top_p"],
                    do_sample=True,
                    repetition_penalty=config["repetition_penalty"],
                    stop_strings=["```"],
                    tokenizer=tokenizer,
                    pad_token_id=tokenizer.eos_token_id,
                )
                new_tokens = outputs[:, prompt_token_length:]
                response = tokenizer.decode(new_tokens[0], skip_special_tokens=True)
                parsed = parse_json_response(response.strip())
                if parsed is not None and is_valid_record(parsed):
                    predictions.append(parsed)
                    with open(output_file, "w", encoding="utf-8") as f:
                        json.dump(predictions, f, indent=2, ensure_ascii=False)
                else:
                    failed_responses.append(response.strip())
                    with open(debug_file, "w", encoding="utf-8") as f:
                        json.dump(failed_responses, f, indent=2, ensure_ascii=False)
    total_attempts = len(predictions) + len(failed_responses)
    print(f"Done: {len(predictions)} valid in {total_attempts} attempts ({len(failed_responses)} discarded)")
    return predictions

output_file = f"outputs/{config['run_name']}.json"
predictions = generate_samples(model, tokenizer, prompt_tokens, config, output_file)
torch.cuda.empty_cache()
print(f"Results saved to: {output_file}")
print("\n===========================| Generate Synthetic Data completed. |===")